# 21. Preference Alignment Features — Facial Skincare

This notebook constructs one ranking-outcome-independent feature row per case from the fixed synthetic query, strict pre-target history, training-safe QCHS history, and item-facet provenance. Functional alignment uses catalogue-relative IDF weighting. Target-profile coverage is directional, Brand remains a separate channel, and catalogue-derived and historical-review-derived facet evidence remain distinct.

The primary feature export is created without loading ranks, predictions, NDCG, or other performance outcomes. Two final, explicitly separated diagnostic joins then examine top-five catalogue-facet alignment and the association between the frozen alignment features and canonical RankP − Base deltas. These joins do not alter the feature table or its definitions.

The outputs support descriptive mechanism analysis only. Alignment is not a relevance label, causal treatment, or basis for model selection.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ==== Imports ====
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import json
import math
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 220)


In [3]:
# ==== Config ====
NOTEBOOK_NAME = "24_preference_alignment_face.ipynb"
CATEGORY_ID = "face"
CATEGORY_LABEL = "Facial Skincare"
REVISION_DATE = "2026-07-17"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")

QUERY_CACHE_PATH = PROJECT_ROOT / "outputs/query_cache/face_queries.parquet"
QUERY_CONTRACT_PATH = PROJECT_ROOT / "outputs/query_summary/face_queries_config.json"
RAW_REVIEWS_PATH = PROJECT_ROOT / "data/raw/reviews_Skin_Care_Face_W2_2019_2023.parquet"
STRICT_HISTORY_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_user_prior_review_history.parquet"
TRAINING_HISTORY_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_user_prior_review_history_training.parquet"
SAMPLING_MANIFEST_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_user_regime_sampling_manifest.json"
ITEM_FACETS_PATH = PROJECT_ROOT / "data/processed/items/face_items_facets.parquet"
ITEM_DOCS_PATH = PROJECT_ROOT / "data/processed/items/face_item_docs.parquet"
RETRIEVAL_MANIFEST_PATH = PROJECT_ROOT / "data/processed/items/retrieval_artifact_manifest_face.json"
PROFILE_DIAGNOSTICS_PATH = PROJECT_ROOT / "outputs/stage1_personalized_retrieval/personalized_retrieval_user_profile_diagnostics_face.csv"
PERSONALIZED_MANIFEST_PATH = PROJECT_ROOT / "outputs/stage1_personalized_retrieval/personalized_retrieval_manifest_face.json"

STRICT_HISTORY_MANIFEST_KEY = "prior_history_parquet"
TRAINING_HISTORY_MANIFEST_KEY = "training_prior_history_parquet"
ITEM_FACETS_MANIFEST_POINTER = ('items_facets_path',)
ITEM_DOCS_MANIFEST_POINTER = ('item_docs_path',)

OUT_DIR = PROJECT_ROOT / "outputs/analysis/preference_alignment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILES = {
    "case_alignment_features": OUT_DIR / "case_alignment_features.parquet",
    "alignment_definition_manifest": OUT_DIR / "alignment_definition_manifest.json",
    "alignment_join_qc": OUT_DIR / "alignment_join_qc.csv",
    "facet_coverage_summary": OUT_DIR / "facet_coverage_summary.csv",
    "profile_and_qchs_coverage_summary": OUT_DIR / "profile_and_qchs_coverage_summary.csv",
}

FACET_FAMILIES = [
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
]
REVIEW_SUFFIX_TO_FAMILY = {
    "concern": "need_benefit_concern",
    "skin_type": "target_context",
    "benefit": "need_benefit_concern",
    "ingredient": "ingredient_or_composition",
    "product_type_or_form_texture": "form_texture",
    "product_form_texture": "form_texture",
    "product_type": "category_or_product_type",
    "form": "form_texture",
    "formulation": "ingredient_or_composition",
    "texture": "form_texture",
    "usage_target": "target_context",
    "claim_diet": "claim_constraint",
    "flavor": "sensory",
    "scent": "sensory",
}
TOKEN_EQUIVALENCE = {'acne': 'acne',
 'blemish': 'acne',
 'blemishes': 'acne',
 'breakout': 'acne',
 'breakouts': 'acne',
 'brighten': 'brightening',
 'brightening': 'brightening',
 'brightness': 'brightening',
 'cleanser': 'cleanser',
 'cleansers': 'cleanser',
 'cleansing': 'cleanser',
 'cream': 'moisturizer',
 'creams': 'moisturizer',
 'dark': 'dark',
 'dry': 'dryness',
 'dryness': 'dryness',
 'hydrate': 'hydration',
 'hydrated': 'hydration',
 'hydrating': 'hydration',
 'hydration': 'hydration',
 'hyperpigmentation': 'dark_spots',
 'lotion': 'moisturizer',
 'lotions': 'moisturizer',
 'mask': 'mask',
 'masks': 'mask',
 'moisturize': 'moisturizer',
 'moisturizer': 'moisturizer',
 'moisturizers': 'moisturizer',
 'moisturizing': 'moisturizer',
 'oiliness': 'oiliness',
 'oily': 'oiliness',
 'pigmentation': 'dark_spots',
 'pore': 'pores',
 'pores': 'pores',
 'sensitive': 'sensitivity',
 'sensitivity': 'sensitivity',
 'serum': 'serum',
 'serums': 'serum',
 'spot': 'dark_spots',
 'spots': 'dark_spots',
 'toner': 'toner',
 'toners': 'toner',
 'treatment': 'treatment',
 'treatments': 'treatment',
 'wash': 'cleanser',
 'wrinkle': 'wrinkles',
 'wrinkles': 'wrinkles'}
REGIME_ORDER = ['cold', 'weak', 'moderate', 'strong']
REGIME_COUNT_RULES = {'cold': (0, 0), 'weak': (1, 4), 'moderate': (5, 9), 'strong': (10, None)}

RECENT_WINDOW_DAYS = 180
PRIMARY_FUNCTIONAL_COMPOSITE = "query_profile_functional_composite"
OUTCOME_COLUMNS_LOADED = []

print("Category:", CATEGORY_LABEL)
print("Query cache:", QUERY_CACHE_PATH)
print("Output directory:", OUT_DIR)


Category: Facial Skincare
Query cache: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_cache/face_queries.parquet
Output directory: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/preference_alignment


## Fixed Alignment Definitions
For eligible facet key \(f\), catalog-relative inverse document frequency is
\[
\operatorname{IDF}(f)=\log\left(\frac{1+N}{1+\operatorname{df}(f)}\right)+1.
\]
For facet sets \(A\) and \(B\), weighted Jaccard is
\[
J_w(A,B)=\frac{\sum_{f\in A\cap B}\operatorname{IDF}(f)}{\sum_{f\in A\cup B}\operatorname{IDF}(f)}.
\]
Target-facet coverage is directional: the numerator is the target/profile intersection and the denominator is target IDF mass. Query facets are matched by contiguous normalized facet-token sequences. Brand is a separate channel. Catalog-derived and historical global-review-derived query-target evidence remain separate. Within this diagnostic, `query_profile_functional_composite` is the fixed primary query-history coherence measure. It aggregates functional families without outcome-based tuning; Brand remains a separate channel. These quantities are mechanism indicators rather than relevance labels.

In [4]:
# ==== Helpers ====
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def boolean_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    text = series.astype("string").str.strip().str.lower()
    numeric = pd.to_numeric(series, errors="coerce")
    mapped = text.map({
        "true": True,
        "false": False,
        "yes": True,
        "no": False,
        "1": True,
        "0": False,
    })
    return mapped.where(mapped.notna(), numeric.map({1.0: True, 0.0: False})).fillna(False).astype(bool)


def require_columns(df, required, label):
    missing = sorted(set(required).difference(df.columns))
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def require_unique_columns(df, label):
    duplicated = df.columns[df.columns.duplicated()].astype(str).tolist()
    if duplicated:
        raise RuntimeError(f"{label} has duplicated columns: {duplicated}")


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def parquet_columns(path):
    return list(pq.ParquetFile(path).schema.names)


def first_existing(columns, candidates, label):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    raise RuntimeError(f"Missing {label}. Tried: {candidates}")


def to_timestamp_ms(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.dropna().empty:
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)
    if numeric.dropna().max() < 10**11:
        numeric = numeric * 1000
    return numeric


def canonical_tokens(text):
    raw_tokens = re.findall(r"[a-z0-9]+", normalize_space(text).lower())
    return [TOKEN_EQUIVALENCE.get(token, token) for token in raw_tokens]


def contains_token_sequence(tokens, phrase_tokens):
    if not phrase_tokens or len(phrase_tokens) > len(tokens):
        return False
    width = len(phrase_tokens)
    return any(tuple(tokens[start:start + width]) == tuple(phrase_tokens) for start in range(len(tokens) - width + 1))


def make_phrase_index(facet_frame):
    index = defaultdict(list)
    for row in facet_frame[["facet_key", "facet_value_norm"]].drop_duplicates().itertuples(index=False):
        phrase_tokens = tuple(canonical_tokens(row.facet_value_norm))
        if phrase_tokens:
            index[phrase_tokens[0]].append((row.facet_key, phrase_tokens))
    for token in index:
        index[token] = sorted(index[token], key=lambda value: (len(value[1]), value[0]))
    return dict(index)


def matched_facet_keys(text, phrase_index):
    tokens = canonical_tokens(text)
    matched = set()
    for start, token in enumerate(tokens):
        for facet_key, phrase_tokens in phrase_index.get(token, []):
            width = len(phrase_tokens)
            if tuple(tokens[start:start + width]) == phrase_tokens:
                matched.add(facet_key)
    return matched


def weighted_jaccard(left_keys, right_keys, idf_lookup):
    left = set(left_keys)
    right = set(right_keys)
    union = left | right
    if not union:
        return np.nan
    denominator = sum(float(idf_lookup[key]) for key in union)
    numerator = sum(float(idf_lookup[key]) for key in left & right)
    return float(numerator / denominator) if denominator > 0 else np.nan


def weighted_coverage(target_keys, profile_keys, idf_lookup):
    target = set(target_keys)
    if not target:
        return np.nan
    denominator = sum(float(idf_lookup[key]) for key in target)
    numerator = sum(float(idf_lookup[key]) for key in target & set(profile_keys))
    return float(numerator / denominator) if denominator > 0 else np.nan


def distribution_stats(weight_map):
    values = np.asarray([float(value) for value in weight_map.values() if float(value) > 0], dtype=float)
    if values.size == 0:
        return {
            "breadth": 0,
            "dominant_share": np.nan,
            "hhi": np.nan,
            "entropy_normalized": np.nan,
        }
    probabilities = values / values.sum()
    entropy = float(-np.sum(probabilities * np.log(probabilities)))
    entropy_normalized = 0.0 if len(probabilities) == 1 else float(entropy / math.log(len(probabilities)))
    return {
        "breadth": int(len(probabilities)),
        "dominant_share": float(probabilities.max()),
        "hhi": float(np.square(probabilities).sum()),
        "entropy_normalized": entropy_normalized,
    }


def safe_rate(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan


def split_pipe(value):
    text = normalize_space(value)
    if not text:
        return []
    return [part.strip() for part in text.split("|") if part.strip()]


def review_semantic_family(facet_type):
    facet_type = normalize_space(facet_type).lower()
    for prefix in ("review_reputation_", "historical_review_"):
        if facet_type.startswith(prefix):
            facet_type = facet_type[len(prefix):]
            break
    return REVIEW_SUFFIX_TO_FAMILY.get(facet_type)


def require_manifest_path(manifest, pointer, expected_path, label):
    observed = manifest
    for key in pointer:
        if not isinstance(observed, dict) or key not in observed:
            dotted_pointer = ".".join(pointer)
            raise RuntimeError(f"{label} does not expose {dotted_pointer!r}.")
        observed = observed[key]
    observed_path = Path(observed)
    if observed_path != Path(expected_path):
        raise RuntimeError(
            f"{label} path mismatch for {'.'.join(pointer)}: "
            f"observed={observed_path}, expected={expected_path}"
        )


def read_history(path, label):
    schema = parquet_columns(path)
    prior_item_column = first_existing(
        schema,
        ["prior_item_id", "prior_parent_asin"],
        f"{label} prior item column",
    )
    columns = [
        "case_id",
        "user_id",
        "target_timestamp_ms",
        prior_item_column,
        "prior_timestamp_ms",
    ]
    if "target_parent_asin" in schema:
        columns.append("target_parent_asin")
    frame = pd.read_parquet(path, columns=list(dict.fromkeys(columns))).copy()
    if prior_item_column != "prior_item_id":
        frame = frame.rename(columns={prior_item_column: "prior_item_id"})
    require_unique_columns(frame, label)
    return frame


def normalize_history(frame, queries_df, label):
    frame = frame.copy()
    for column in ["case_id", "user_id", "prior_item_id"]:
        frame[column] = frame[column].fillna("").astype(str).map(normalize_space)
    for column in ["target_timestamp_ms", "prior_timestamp_ms"]:
        frame[column] = pd.to_numeric(frame[column], errors="raise").astype("int64")

    query_case_ids = set(queries_df["case_id"])
    extra_case_count = int(frame.loc[~frame["case_id"].isin(query_case_ids), "case_id"].nunique())
    frame = frame.loc[frame["case_id"].isin(query_case_ids) & frame["prior_item_id"].ne("")].copy()

    query_users = queries_df.set_index("case_id")["user_id"]
    query_targets = queries_df.set_index("case_id")["target_parent_asin"]
    query_timestamps = queries_df.set_index("case_id")["target_timestamp_ms"]
    if len(frame):
        if not frame["user_id"].eq(frame["case_id"].map(query_users)).all():
            raise RuntimeError(f"{label} user_id does not match the query case.")
        if not frame["target_timestamp_ms"].eq(frame["case_id"].map(query_timestamps)).all():
            raise RuntimeError(f"{label} target timestamp does not match the query case.")
        if "target_parent_asin" in frame.columns:
            frame["target_parent_asin"] = frame["target_parent_asin"].fillna("").astype(str).map(normalize_space)
            if not frame["target_parent_asin"].eq(frame["case_id"].map(query_targets)).all():
                raise RuntimeError(f"{label} target item does not match the query case.")
        else:
            frame["target_parent_asin"] = frame["case_id"].map(query_targets)
    else:
        frame["target_parent_asin"] = pd.Series(dtype="object")
    return frame, extra_case_count


def regime_from_count(prior_count):
    count = int(prior_count)
    for regime, bounds in REGIME_COUNT_RULES.items():
        lower, upper = bounds
        if count >= lower and (upper is None or count <= upper):
            return regime
    raise RuntimeError(f"No regime rule for prior count: {count}")


def keys_for_family(keys, family):
    prefix = f"{family}::"
    return {key for key in keys if str(key).startswith(prefix)}


def mean_or_nan(series):
    values = pd.to_numeric(series, errors="coerce")
    return float(values.mean()) if values.notna().any() else np.nan


def median_or_nan(series):
    values = pd.to_numeric(series, errors="coerce")
    return float(values.median()) if values.notna().any() else np.nan


def qc_row(check_name, status, observed, expected, details=""):
    return {
        "check_name": check_name,
        "status": status,
        "observed_value": observed,
        "expected_value": expected,
        "details": details,
    }


In [5]:
# ==== Load Inputs and Validate Contracts ====
required_paths = [
    QUERY_CACHE_PATH,
    QUERY_CONTRACT_PATH,
    RAW_REVIEWS_PATH,
    STRICT_HISTORY_PATH,
    TRAINING_HISTORY_PATH,
    SAMPLING_MANIFEST_PATH,
    ITEM_FACETS_PATH,
    ITEM_DOCS_PATH,
    RETRIEVAL_MANIFEST_PATH,
    PROFILE_DIAGNOSTICS_PATH,
    PERSONALIZED_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required Notebook 03/04/06/08 artifacts: {missing_paths}")

sampling_manifest = load_json(SAMPLING_MANIFEST_PATH)
retrieval_manifest = load_json(RETRIEVAL_MANIFEST_PATH)
query_contract = load_json(QUERY_CONTRACT_PATH)
personalized_manifest = load_json(PERSONALIZED_MANIFEST_PATH)

candidate_fallback_contract = {
    "cold_baseline_fallback_policy": "exact_query_only_winner_copy",
    "fallback_policy": "copy_baseline_candidates",
    "fallback_changes_case_universe": False,
    "fallback_uses_all_prior": False,
}
for key, expected_value in candidate_fallback_contract.items():
    if personalized_manifest.get(key) != expected_value:
        raise RuntimeError(
            f"Notebook 08 candidate fallback contract mismatch for {key}: "
            f"observed={personalized_manifest.get(key)!r}, expected={expected_value!r}"
        )

require_manifest_path(
    sampling_manifest,
    ("output_paths", STRICT_HISTORY_MANIFEST_KEY),
    STRICT_HISTORY_PATH,
    "Notebook 03 manifest",
)
require_manifest_path(
    sampling_manifest,
    ("output_paths", TRAINING_HISTORY_MANIFEST_KEY),
    TRAINING_HISTORY_PATH,
    "Notebook 03 manifest",
)
require_manifest_path(
    retrieval_manifest,
    ITEM_FACETS_MANIFEST_POINTER,
    ITEM_FACETS_PATH,
    "Notebook 04 manifest",
)
require_manifest_path(
    retrieval_manifest,
    ITEM_DOCS_MANIFEST_POINTER,
    ITEM_DOCS_PATH,
    "Notebook 04 manifest",
)
require_manifest_path(
    query_contract,
    ("output_paths", "query_cache"),
    QUERY_CACHE_PATH,
    "Notebook 06 contract",
)
require_manifest_path(
    personalized_manifest,
    ("output_paths", "profile_diagnostics"),
    PROFILE_DIAGNOSTICS_PATH,
    "Notebook 08 manifest",
)

required_query_columns = [
    "case_id",
    "user_id",
    "regime",
    "target_parent_asin",
    "target_timestamp_ms",
    "query",
    "safe_signal_count",
    "query_specific_facet_family_count",
]
optional_query_columns = ["brand_or_name_leak_flag"]

query_schema = pq.ParquetFile(QUERY_CACHE_PATH).schema.names
query_columns = required_query_columns + [
    column for column in optional_query_columns if column in query_schema
]

queries = pd.read_parquet(QUERY_CACHE_PATH, columns=query_columns).copy()
require_columns(queries, required_query_columns, "Notebook 06 query cache")

if "brand_or_name_leak_flag" in queries.columns:
    query_brand_or_name_leak_flag_source = "notebook06_query_cache"
else:
    queries["brand_or_name_leak_flag"] = False
    query_brand_or_name_leak_flag_source = "not_present_in_notebook06_query_cache"
require_unique_columns(queries, "Notebook 06 query cache")
for column in ["case_id", "user_id", "regime", "target_parent_asin", "query"]:
    queries[column] = queries[column].fillna("").astype(str).map(normalize_space)
queries["target_timestamp_ms"] = pd.to_numeric(
    queries["target_timestamp_ms"], errors="raise"
).astype("int64")
queries["safe_signal_count"] = pd.to_numeric(queries["safe_signal_count"], errors="raise").astype(int)
queries["query_specific_facet_family_count"] = pd.to_numeric(
    queries["query_specific_facet_family_count"], errors="raise"
).astype(int)
queries["brand_or_name_leak_flag"] = boolean_series(queries["brand_or_name_leak_flag"])

if queries["case_id"].eq("").any() or queries["case_id"].duplicated().any():
    raise RuntimeError("Notebook 06 must contain one non-empty row per case_id.")
if queries["user_id"].eq("").any() or queries["user_id"].duplicated().any():
    raise RuntimeError("Notebook 06 must contain one case per user_id.")
if queries[["target_parent_asin", "query"]].eq("").any().any():
    raise RuntimeError("Notebook 06 contains an empty target item or query.")
if queries["brand_or_name_leak_flag"].any():
    raise RuntimeError("Notebook 06 marks at least one active query with brand/name leakage.")
if set(queries["regime"]) != set(REGIME_ORDER):
    raise RuntimeError(f"Unexpected query regimes: {sorted(set(queries['regime']))}")

facet_columns = [
    "parent_asin",
    "facet_type",
    "facet_value_norm",
    "facet_role",
    "is_brand",
    "is_review_derived",
    "is_product_functional_facet",
    "is_query_safe",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_retrieval_safe",
    "is_profile_safe",
]
facets = pd.read_parquet(ITEM_FACETS_PATH, columns=facet_columns).copy()
require_columns(facets, facet_columns, "Notebook 04 item facets")
require_unique_columns(facets, "Notebook 04 item facets")

item_docs = pd.read_parquet(ITEM_DOCS_PATH, columns=["parent_asin"]).copy()
require_columns(item_docs, ["parent_asin"], "Notebook 04 item docs")
require_unique_columns(item_docs, "Notebook 04 item docs")
item_docs["parent_asin"] = item_docs["parent_asin"].fillna("").astype(str).map(normalize_space)
item_docs = item_docs.loc[item_docs["parent_asin"].ne("")].copy()
if item_docs.empty or item_docs["parent_asin"].duplicated().any():
    raise RuntimeError("Notebook 04 item docs must contain one non-empty row per parent_asin.")

profile_columns = [
    "case_id",
    "user_id",
    "regime",
    "target_parent_asin",
    "prior_review_event_count",
    "prior_unique_item_count",
    "stage1_usable_prior_item_count",
    "stage1_profile_safe_prior_item_count",
    "stage1_profile_available",
    "qchs_selected_prior_item_count",
    "qchs_selected_ratio",
    "qchs_alignment_mean",
    "qchs_alignment_max",
    "qchs_attention_entropy",
    "qchs_profile_safe_facet_count",
    "qchs_brand_facet_count",
    "qchs_functional_facet_count",
    "qchs_anchor_phrase_count",
    "qchs_profile_available",
    "profile_fallback_flag",
    "profile_fallback_reason",
    "qchs_fallback_flag",
    "qchs_selected_prior_items",
]
# Compatibility shim for legacy Notebook 08 QCHA diagnostics.
actual_columns = pd.read_csv(PROFILE_DIAGNOSTICS_PATH, nrows=0).columns.tolist()

legacy_to_current = {
    "qcha_selected_prior_item_count": "qchs_selected_prior_item_count",
    "qcha_selected_ratio": "qchs_selected_ratio",
    "qcha_alignment_mean": "qchs_alignment_mean",
    "qcha_alignment_max": "qchs_alignment_max",
    "qcha_attention_entropy": "qchs_attention_entropy",
    "qcha_profile_safe_facet_count": "qchs_profile_safe_facet_count",
    "qcha_brand_facet_count": "qchs_brand_facet_count",
    "qcha_functional_facet_count": "qchs_functional_facet_count",
    "qcha_anchor_phrase_count": "qchs_anchor_phrase_count",
    "qcha_profile_available": "qchs_profile_available",
    "qcha_fallback_flag": "qchs_fallback_flag",
    "qcha_selected_prior_items": "qchs_selected_prior_items",
}

read_columns = set()
missing = []

for col in profile_columns:
    if col in actual_columns:
        read_columns.add(col)
    else:
        legacy_col = next(
            (old for old, new in legacy_to_current.items() if new == col and old in actual_columns),
            None,
        )
        if legacy_col is None:
            missing.append(col)
        else:
            read_columns.add(legacy_col)

if missing:
    raise RuntimeError(f"Missing profile diagnostics columns: {missing}")

profile_diagnostics = pd.read_csv(PROFILE_DIAGNOSTICS_PATH, usecols=sorted(read_columns)).copy()
profile_diagnostics = profile_diagnostics.rename(columns=legacy_to_current)
require_columns(profile_diagnostics, profile_columns, "Notebook 08 profile diagnostics")
require_unique_columns(profile_diagnostics, "Notebook 08 profile diagnostics")
for column in ["case_id", "user_id", "regime", "target_parent_asin"]:
    profile_diagnostics[column] = profile_diagnostics[column].fillna("").astype(str).map(normalize_space)
for column in [
    "stage1_profile_available",
    "qchs_profile_available",
    "profile_fallback_flag",
    "qchs_fallback_flag",
]:
    profile_diagnostics[column] = boolean_series(profile_diagnostics[column])
profile_integer_columns = [
    "prior_review_event_count",
    "prior_unique_item_count",
    "stage1_usable_prior_item_count",
    "stage1_profile_safe_prior_item_count",
    "qchs_selected_prior_item_count",
    "qchs_profile_safe_facet_count",
    "qchs_brand_facet_count",
    "qchs_functional_facet_count",
    "qchs_anchor_phrase_count",
]
for column in profile_integer_columns:
    profile_diagnostics[column] = pd.to_numeric(
        profile_diagnostics[column], errors="raise"
    ).astype(int)
profile_float_columns = [
    "qchs_selected_ratio",
    "qchs_alignment_mean",
    "qchs_alignment_max",
    "qchs_attention_entropy",
]
for column in profile_float_columns:
    profile_diagnostics[column] = pd.to_numeric(
        profile_diagnostics[column], errors="raise"
    ).astype(float)
profile_diagnostics["profile_fallback_reason"] = (
    profile_diagnostics["profile_fallback_reason"].fillna("").astype(str).map(normalize_space)
)
profile_diagnostics["qchs_selected_prior_items"] = (
    profile_diagnostics["qchs_selected_prior_items"].fillna("").astype(str).map(normalize_space)
)

if profile_diagnostics["case_id"].eq("").any() or profile_diagnostics["case_id"].duplicated().any():
    raise RuntimeError("Notebook 08 profile diagnostics must contain one non-empty row per case_id.")
if set(profile_diagnostics["case_id"]) != set(queries["case_id"]):
    raise RuntimeError("Notebook 08 profile diagnostics and Notebook 06 query cache have different case universes.")
cold_profile_rows = profile_diagnostics.loc[profile_diagnostics["regime"].eq("cold")]
if (
    cold_profile_rows["qchs_profile_available"].any()
    or cold_profile_rows["qchs_selected_prior_item_count"].ne(0).any()
    or not cold_profile_rows["profile_fallback_flag"].all()
):
    raise RuntimeError("Notebook 08 cold cases must have zero QCHS selections and exact baseline fallback.")

print("Cases:", len(queries))
print("Item-facet rows:", len(facets))
print("Profile diagnostic rows:", len(profile_diagnostics))
print("Validation: upstream paths and one-to-one source grains passed")


Cases: 2288
Item-facet rows: 984052
Profile diagnostic rows: 2288
Validation: upstream paths and one-to-one source grains passed


In [6]:
# ==== Functional, Review-Derived, and Brand Facet Indexes ====
boolean_facet_columns = [
    "is_brand",
    "is_review_derived",
    "is_product_functional_facet",
    "is_query_safe",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_retrieval_safe",
    "is_profile_safe",
]
for column in boolean_facet_columns:
    facets[column] = boolean_series(facets[column])
facets["parent_asin"] = facets["parent_asin"].fillna("").astype(str).map(normalize_space)
for column in ["facet_type", "facet_value_norm", "facet_role"]:
    facets[column] = facets[column].fillna("").astype(str).map(normalize_space).str.lower()

catalog_item_ids = set(item_docs["parent_asin"])
facet_item_ids = set(facets.loc[facets["parent_asin"].ne(""), "parent_asin"])
if not facet_item_ids.issubset(catalog_item_ids):
    raise RuntimeError("Notebook 04 item facets contain parent_asin values absent from item docs.")

catalog_functional_mask = (
    facets["is_product_functional_facet"]
    & facets["facet_value_norm"].ne("")
    & facets["is_query_safe"]
    & ~facets["is_brand"]
    & ~facets["is_review_derived"]
    & facets["is_metadata_facet_source"]
    & ~facets["is_disallowed_nonfacet_source"]
    & ~facets["is_generic_category_anchor"]
    & ~facets["is_generic_utility_token"]
    & ~facets["is_context_dependent_utility_token"]
)
brand_mask = (
    facets["is_brand"]
    & facets["facet_value_norm"].ne("")
    & facets["is_profile_safe"]
    & ~facets["is_review_derived"]
)
review_derived_mask = (
    facets["is_review_derived"]
    & facets["facet_value_norm"].ne("")
    & facets["is_retrieval_safe"]
    & ~facets["is_brand"]
)

catalog_functional_facets = facets.loc[
    catalog_functional_mask,
    ["parent_asin", "facet_role", "facet_value_norm"],
].rename(columns={"facet_role": "semantic_family"})
if catalog_functional_facets.empty:
    raise RuntimeError("No catalog-derived product-functional facets are available.")
if not catalog_functional_facets["semantic_family"].isin(FACET_FAMILIES).all():
    invalid_families = sorted(
        set(catalog_functional_facets.loc[
            ~catalog_functional_facets["semantic_family"].isin(FACET_FAMILIES),
            "semantic_family",
        ])
    )
    raise RuntimeError(f"Unexpected catalog functional families: {invalid_families}")

brand_facets = facets.loc[
    brand_mask,
    ["parent_asin", "facet_value_norm"],
].copy()
if brand_facets.empty:
    raise RuntimeError("Brand facets are required as a separate profile channel.")
brand_facets["semantic_family"] = "brand"

review_derived_facets = facets.loc[
    review_derived_mask,
    ["parent_asin", "facet_type", "facet_value_norm"],
].copy()
review_derived_facets["semantic_family"] = review_derived_facets["facet_type"].map(
    review_semantic_family
)
unmapped_review_derived_rows = int(review_derived_facets["semantic_family"].isna().sum())
review_derived_facets = review_derived_facets.loc[
    review_derived_facets["semantic_family"].isin(FACET_FAMILIES)
].copy()

for frame in [catalog_functional_facets, brand_facets, review_derived_facets]:
    frame["facet_key"] = frame["semantic_family"] + "::" + frame["facet_value_norm"]
    frame.drop_duplicates(["parent_asin", "facet_key"], inplace=True)

catalog_item_count = int(len(catalog_item_ids))
catalog_df = catalog_functional_facets.groupby("facet_key")["parent_asin"].nunique()
catalog_idf = {
    key: float(math.log((1.0 + catalog_item_count) / (1.0 + frequency)) + 1.0)
    for key, frequency in catalog_df.items()
}
review_df = review_derived_facets.groupby("facet_key")["parent_asin"].nunique()
review_idf = {
    key: float(math.log((1.0 + catalog_item_count) / (1.0 + frequency)) + 1.0)
    for key, frequency in review_df.items()
}

catalog_keys_by_item = {
    item_id: set(group["facet_key"])
    for item_id, group in catalog_functional_facets.groupby("parent_asin", sort=False)
}
brand_keys_by_item = {
    item_id: set(group["facet_key"])
    for item_id, group in brand_facets.groupby("parent_asin", sort=False)
}
review_keys_by_item = {
    item_id: set(group["facet_key"])
    for item_id, group in review_derived_facets.groupby("parent_asin", sort=False)
}
catalog_phrase_index = make_phrase_index(catalog_functional_facets)
review_phrase_index = make_phrase_index(review_derived_facets)
brand_tokens_by_key = {
    row.facet_key: tuple(canonical_tokens(row.facet_value_norm))
    for row in brand_facets[["facet_key", "facet_value_norm"]].drop_duplicates().itertuples(index=False)
}

query_catalog_keys_by_case = {
    row.case_id: matched_facet_keys(row.query, catalog_phrase_index)
    for row in queries[["case_id", "query"]].itertuples(index=False)
}
query_review_keys_by_case = {
    row.case_id: matched_facet_keys(row.query, review_phrase_index)
    for row in queries[["case_id", "query"]].itertuples(index=False)
}

if catalog_functional_facets["semantic_family"].eq("brand").any():
    raise RuntimeError("Brand entered the catalog functional facet channel.")
if facets.loc[catalog_functional_mask, "is_review_derived"].any():
    raise RuntimeError("Review-derived rows entered the user-profile functional channel.")

print("Catalog functional facet keys:", len(catalog_idf))
print("Mapped review-derived facet keys:", len(review_idf))
print("Brand facet keys:", brand_facets["facet_key"].nunique())
print("Unmapped review-derived rows retained only in QC:", unmapped_review_derived_rows)
print("Validation: functional and brand channels are separate")


Catalog functional facet keys: 9987
Mapped review-derived facet keys: 44
Brand facet keys: 17871
Unmapped review-derived rows retained only in QC: 0
Validation: functional and brand channels are separate


In [7]:
# ==== Strict and Training-Safe History Lineage ====
raw_schema = parquet_columns(RAW_REVIEWS_PATH)
raw_user_column = first_existing(
    raw_schema,
    ["user_id", "reviewerID", "reviewer_id", "customer_id"],
    "raw review user id",
)
raw_timestamp_column = first_existing(
    raw_schema,
    ["timestamp_ms", "timestamp", "unixReviewTime", "review_time"],
    "raw review timestamp",
)
raw_reviews = pd.read_parquet(
    RAW_REVIEWS_PATH,
    columns=list(dict.fromkeys([raw_user_column, "parent_asin", raw_timestamp_column])),
).copy()
raw_reviews = pd.DataFrame({
    "user_id": raw_reviews[raw_user_column].fillna("").astype(str).map(normalize_space),
    "prior_item_id": raw_reviews["parent_asin"].fillna("").astype(str).map(normalize_space),
    "prior_timestamp_ms": to_timestamp_ms(raw_reviews[raw_timestamp_column]),
})
raw_reviews = raw_reviews.loc[
    raw_reviews["user_id"].isin(set(queries["user_id"]))
    & raw_reviews["prior_item_id"].ne("")
].dropna(subset=["prior_timestamp_ms"]).copy()
raw_reviews["prior_timestamp_ms"] = raw_reviews["prior_timestamp_ms"].astype("int64")

case_keys = queries[[
    "case_id",
    "user_id",
    "target_parent_asin",
    "target_timestamp_ms",
]].copy()
strict_history = case_keys.merge(
    raw_reviews,
    on="user_id",
    how="left",
    validate="one_to_many",
)
strict_history = strict_history.loc[
    strict_history["prior_timestamp_ms"].notna()
    & strict_history["prior_timestamp_ms"].lt(strict_history["target_timestamp_ms"])
    & strict_history["prior_item_id"].ne(strict_history["target_parent_asin"])
].copy()
strict_history["prior_timestamp_ms"] = strict_history["prior_timestamp_ms"].astype("int64")
strict_history = strict_history.sort_values(
    ["case_id", "prior_timestamp_ms", "prior_item_id"],
    ascending=[True, False, True],
    kind="mergesort",
).reset_index(drop=True)

strict_history_artifact_all = read_history(STRICT_HISTORY_PATH, "Notebook 03 strict history")
strict_history_artifact, strict_history_extra_case_count = normalize_history(
    strict_history_artifact_all,
    queries,
    "Notebook 03 strict history",
)
training_history_all = read_history(TRAINING_HISTORY_PATH, "Notebook 03 training-safe history")
training_history, training_history_extra_case_count = normalize_history(
    training_history_all,
    queries,
    "Notebook 03 training-safe history",
)

for label, frame in [
    ("reconstructed strict history", strict_history),
    ("Notebook 03 strict history", strict_history_artifact),
    ("Notebook 03 training-safe history", training_history),
]:
    if len(frame) and not frame["prior_timestamp_ms"].lt(frame["target_timestamp_ms"]).all():
        raise RuntimeError(f"{label} contains a target or future interaction.")
    if len(frame) and frame["prior_item_id"].eq(frame["target_parent_asin"]).any():
        raise RuntimeError(f"{label} contains the held-out target item.")

event_keys = [
    "case_id",
    "user_id",
    "target_timestamp_ms",
    "prior_item_id",
    "prior_timestamp_ms",
]
strict_artifact_case_ids = set(strict_history_artifact["case_id"])
strict_artifact_counts = (
    strict_history_artifact.groupby(event_keys, dropna=False).size().rename("artifact_count").reset_index()
)
strict_reconstructed_overlap_counts = (
    strict_history.loc[strict_history["case_id"].isin(strict_artifact_case_ids)]
    .groupby(event_keys, dropna=False)
    .size()
    .rename("reconstructed_count")
    .reset_index()
)
strict_reconciliation = strict_artifact_counts.merge(
    strict_reconstructed_overlap_counts,
    on=event_keys,
    how="outer",
).fillna(0)
strict_reconciliation_mismatch = strict_reconciliation.loc[
    strict_reconciliation["artifact_count"].ne(strict_reconciliation["reconstructed_count"])
].copy()
if len(strict_reconciliation_mismatch):
    raise RuntimeError(
        "Notebook 03 strict-history rows do not match identifier/timestamp reconstruction for overlapping cases: "
        + json.dumps(strict_reconciliation_mismatch.head(10).to_dict("records"))
    )

strict_counts = strict_history.groupby("case_id").size()
queries["strict_prior_count_reconstructed"] = queries["case_id"].map(strict_counts).fillna(0).astype(int)
queries["regime_reconstructed"] = queries["strict_prior_count_reconstructed"].map(regime_from_count)
if not queries["regime"].eq(queries["regime_reconstructed"]).all():
    mismatch = queries.loc[
        ~queries["regime"].eq(queries["regime_reconstructed"]),
        ["case_id", "regime", "strict_prior_count_reconstructed", "regime_reconstructed"],
    ].head(10)
    raise RuntimeError("Regime does not match reconstructed strict history: " + json.dumps(mismatch.to_dict("records")))

strict_event_counts = (
    strict_history.groupby(event_keys, dropna=False).size().rename("strict_count").reset_index()
)
training_event_counts = (
    training_history.groupby(event_keys, dropna=False).size().rename("training_count").reset_index()
)
training_subset_check = training_event_counts.merge(
    strict_event_counts,
    on=event_keys,
    how="left",
)
training_subset_violations = training_subset_check.loc[
    training_subset_check["training_count"].gt(training_subset_check["strict_count"].fillna(0))
].copy()
if len(training_subset_violations):
    raise RuntimeError(
        "Training-safe history is not a multiset subset of strict pre-target history: "
        + json.dumps(training_subset_violations.head(10).to_dict("records"))
    )

print("Reconstructed strict-history rows:", len(strict_history))
print("Strict-history artifact rows on final cases:", len(strict_history_artifact))
print("Training-safe history rows on final cases:", len(training_history))
print("Validation: target/future exclusion and history lineage passed")


Reconstructed strict-history rows: 29225
Strict-history artifact rows on final cases: 29225
Training-safe history rows on final cases: 22540
Validation: target/future exclusion and history lineage passed


In [8]:
# ==== Notebook 08 QCHS Lineage ====
identity_check = queries[["case_id", "user_id", "regime", "target_parent_asin"]].merge(
    profile_diagnostics[["case_id", "user_id", "regime", "target_parent_asin"]],
    on="case_id",
    how="inner",
    suffixes=("_query", "_qchs"),
    validate="one_to_one",
)
for column in ["user_id", "regime", "target_parent_asin"]:
    if not identity_check[f"{column}_query"].eq(identity_check[f"{column}_qchs"]).all():
        raise RuntimeError(f"Notebook 06/08 identity mismatch: {column}")

training_catalog = training_history.loc[
    training_history["prior_item_id"].isin(catalog_item_ids)
].copy()
training_catalog_event_counts = training_catalog.groupby("case_id").size()
training_catalog_item_counts = training_catalog.groupby("case_id")["prior_item_id"].nunique()
profile_diagnostics["expected_qchs_input_event_count"] = (
    profile_diagnostics["case_id"].map(training_catalog_event_counts).fillna(0).astype(int)
)
profile_diagnostics["expected_qchs_input_unique_item_count"] = (
    profile_diagnostics["case_id"].map(training_catalog_item_counts).fillna(0).astype(int)
)
if not pd.to_numeric(profile_diagnostics["prior_review_event_count"], errors="raise").astype(int).eq(
    profile_diagnostics["expected_qchs_input_event_count"]
).all():
    raise RuntimeError("Notebook 08 prior event count does not match the training-safe catalog history.")
if not pd.to_numeric(profile_diagnostics["prior_unique_item_count"], errors="raise").astype(int).eq(
    profile_diagnostics["expected_qchs_input_unique_item_count"]
).all():
    raise RuntimeError("Notebook 08 prior unique-item count does not match the training-safe catalog history.")

training_items_by_case = {
    case_id: set(group["prior_item_id"])
    for case_id, group in training_history.groupby("case_id", sort=False)
}
selected_lists = profile_diagnostics["qchs_selected_prior_items"].map(split_pipe)
selected_counts = pd.to_numeric(
    profile_diagnostics["qchs_selected_prior_item_count"], errors="raise"
).astype(int)
qchs_selected_count_matches = selected_lists.map(len).eq(selected_counts)
qchs_selected_unique = selected_lists.map(lambda values: len(values) == len(set(values)))
qchs_selected_lineage_ok = [
    set(values).issubset(training_items_by_case.get(case_id, set()))
    for case_id, values in zip(profile_diagnostics["case_id"], selected_lists)
]
if not qchs_selected_count_matches.all():
    raise RuntimeError("Notebook 08 selected-item string does not match qchs_selected_prior_item_count.")
if not qchs_selected_unique.all():
    raise RuntimeError("Notebook 08 QCHS selected-item list contains duplicates.")
if not all(qchs_selected_lineage_ok):
    bad_cases = profile_diagnostics.loc[
        ~pd.Series(qchs_selected_lineage_ok, index=profile_diagnostics.index),
        "case_id",
    ].head(10).tolist()
    raise RuntimeError(f"QCHS selected an item outside training-safe prior history: {bad_cases}")

qchs_input_unique_count = pd.to_numeric(
    profile_diagnostics["prior_unique_item_count"], errors="raise"
).astype(int)
expected_retention = np.where(
    qchs_input_unique_count.gt(0),
    selected_counts / qchs_input_unique_count.where(qchs_input_unique_count.gt(0), 1),
    0.0,
)
observed_retention = pd.to_numeric(profile_diagnostics["qchs_selected_ratio"], errors="raise")
qchs_retention_matches = np.isclose(
    observed_retention.to_numpy(dtype=float),
    np.asarray(expected_retention, dtype=float),
    rtol=0.0,
    atol=1e-12,
)
if not qchs_retention_matches.all():
    raise RuntimeError("Notebook 08 qchs_selected_ratio does not match selected/input unique items.")
if (
    profile_diagnostics["qchs_profile_available"]
    & profile_diagnostics["profile_fallback_flag"]
).any():
    raise RuntimeError("A case cannot be both QCHS-available and a candidate fallback.")

profile_diagnostics["qchs_selected_item_lineage_ok"] = qchs_selected_lineage_ok
profile_diagnostics["qchs_selected_count_matches"] = qchs_selected_count_matches
profile_diagnostics["qchs_retention_matches"] = qchs_retention_matches

print("QCHS-active cases:", int(profile_diagnostics["qchs_profile_available"].sum()))
print("QCHS candidate fallbacks:", int(profile_diagnostics["profile_fallback_flag"].sum()))
print("Validation: QCHS selection, retention, and fallback lineage passed")


QCHS-active cases: 1513
QCHS candidate fallbacks: 775
Validation: QCHS selection, retention, and fallback lineage passed


In [9]:
# ==== Case-Level Coherence Features ====
MS_PER_DAY = 86_400_000.0
strict_history_by_case = {
    case_id: group.copy()
    for case_id, group in strict_history.groupby("case_id", sort=False)
}
training_history_by_case = {
    case_id: group.copy()
    for case_id, group in training_history.groupby("case_id", sort=False)
}

case_rows = []
for query_row in queries.itertuples(index=False):
    case_id = query_row.case_id
    history = strict_history_by_case.get(case_id, strict_history.iloc[0:0])
    training_case_history = training_history_by_case.get(case_id, training_history.iloc[0:0])
    prior_items = history["prior_item_id"].tolist()
    prior_item_counts = Counter(prior_items)
    item_distribution = distribution_stats(prior_item_counts)

    event_count = int(len(history))
    unique_item_count = int(history["prior_item_id"].nunique()) if event_count else 0
    if event_count:
        ages_days = (
            query_row.target_timestamp_ms - history["prior_timestamp_ms"].to_numpy(dtype="int64")
        ) / MS_PER_DAY
        latest_gap_days = float(np.min(ages_days))
        median_gap_days = float(np.median(ages_days))
        mean_gap_days = float(np.mean(ages_days))
        oldest_gap_days = float(np.max(ages_days))
        recent_180d_share = float(np.mean(ages_days <= RECENT_WINDOW_DAYS))
    else:
        latest_gap_days = np.nan
        median_gap_days = np.nan
        mean_gap_days = np.nan
        oldest_gap_days = np.nan
        recent_180d_share = np.nan

    functional_mass = Counter()
    brand_mass = Counter()
    functional_covered_events = 0
    brand_covered_events = 0
    catalog_covered_events = 0
    for item_id in prior_items:
        if item_id in catalog_item_ids:
            catalog_covered_events += 1
        functional_keys = catalog_keys_by_item.get(item_id, set())
        if functional_keys:
            functional_covered_events += 1
            contribution = 1.0 / len(functional_keys)
            for key in functional_keys:
                functional_mass[key] += contribution
        brand_keys = brand_keys_by_item.get(item_id, set())
        if brand_keys:
            brand_covered_events += 1
            contribution = 1.0 / len(brand_keys)
            for key in brand_keys:
                brand_mass[key] += contribution

    profile_functional_keys = set(functional_mass)
    profile_brand_keys = set(brand_mass)
    functional_distribution = distribution_stats(functional_mass)
    brand_distribution = distribution_stats(brand_mass)

    query_catalog_keys = query_catalog_keys_by_case.get(case_id, set())
    query_review_keys = query_review_keys_by_case.get(case_id, set())
    target_catalog_keys = catalog_keys_by_item.get(query_row.target_parent_asin, set())
    target_review_keys = review_keys_by_item.get(query_row.target_parent_asin, set())
    target_brand_keys = brand_keys_by_item.get(query_row.target_parent_asin, set())

    target_brand_hit_count = sum(
        bool(brand_keys_by_item.get(item_id, set()) & target_brand_keys)
        for item_id in prior_items
    )
    known_brand_event_count = sum(
        bool(brand_keys_by_item.get(item_id, set()))
        for item_id in prior_items
    )
    dominant_brand_keys = set()
    if brand_mass:
        max_brand_mass = max(brand_mass.values())
        dominant_brand_keys = {
            key for key, value in brand_mass.items() if np.isclose(value, max_brand_mass)
        }

    query_tokens = canonical_tokens(query_row.query)
    query_target_brand_matches = {
        key
        for key in target_brand_keys
        if contains_token_sequence(query_tokens, brand_tokens_by_key.get(key, ()))
    }

    row = {
        "category_id": CATEGORY_ID,
        "category_label": CATEGORY_LABEL,
        "case_id": case_id,
        "user_id": query_row.user_id,
        "regime": query_row.regime,
        "target_parent_asin": query_row.target_parent_asin,
        "target_timestamp_ms": int(query_row.target_timestamp_ms),
        "query_text": query_row.query,
        "query_source_signal_count": int(query_row.safe_signal_count),
        "query_specific_family_count_upstream": int(query_row.query_specific_facet_family_count),
        "query_brand_or_name_leak_flag_upstream": bool(query_row.brand_or_name_leak_flag),
        "query_catalog_functional_facet_count": int(len(query_catalog_keys)),
        "query_catalog_functional_family_count": int(len({key.split("::", 1)[0] for key in query_catalog_keys})),
        "query_global_review_derived_facet_count": int(len(query_review_keys)),
        "target_item_in_facet_catalog": bool(query_row.target_parent_asin in catalog_item_ids),
        "target_catalog_functional_facet_count": int(len(target_catalog_keys)),
        "target_global_review_derived_facet_count": int(len(target_review_keys)),
        "strict_prior_interaction_count": event_count,
        "strict_prior_unique_item_count": unique_item_count,
        "strict_prior_repeated_interaction_share": safe_rate(event_count - unique_item_count, event_count),
        "history_repeated_item_concentration": item_distribution["dominant_share"],
        "history_item_hhi": item_distribution["hhi"],
        "history_item_entropy_normalized": item_distribution["entropy_normalized"],
        "strict_prior_latest_gap_days": latest_gap_days,
        "strict_prior_median_gap_days": median_gap_days,
        "strict_prior_mean_gap_days": mean_gap_days,
        "strict_prior_oldest_gap_days": oldest_gap_days,
        "strict_prior_recent_180d_share": recent_180d_share,
        "training_safe_prior_interaction_count": int(len(training_case_history)),
        "training_safe_prior_unique_item_count": int(training_case_history["prior_item_id"].nunique()) if len(training_case_history) else 0,
        "strict_history_item_catalog_coverage_rate": safe_rate(catalog_covered_events, event_count),
        "profile_functional_item_coverage_rate": safe_rate(functional_covered_events, event_count),
        "profile_brand_item_coverage_rate": safe_rate(brand_covered_events, event_count),
        "profile_functional_facet_breadth": int(functional_distribution["breadth"]),
        "profile_functional_family_breadth": int(len({key.split("::", 1)[0] for key in profile_functional_keys})),
        "profile_safe_facet_breadth": int(len(profile_functional_keys) + len(profile_brand_keys)),
        "profile_functional_dominant_share": functional_distribution["dominant_share"],
        "profile_functional_hhi": functional_distribution["hhi"],
        "profile_functional_entropy_normalized": functional_distribution["entropy_normalized"],
        "unique_prior_brand_count": int(brand_distribution["breadth"]),
        "dominant_brand_share": brand_distribution["dominant_share"],
        "brand_hhi": brand_distribution["hhi"],
        "brand_entropy_normalized": brand_distribution["entropy_normalized"],
        "target_brand_available": bool(target_brand_keys),
        "target_brand_seen_in_prior": bool(target_brand_keys & profile_brand_keys),
        "target_brand_prior_interaction_share": (
            safe_rate(target_brand_hit_count, event_count) if target_brand_keys else np.nan
        ),
        "target_brand_prior_known_brand_share": (
            safe_rate(target_brand_hit_count, known_brand_event_count) if target_brand_keys else np.nan
        ),
        "target_brand_is_dominant_prior_brand": bool(target_brand_keys & dominant_brand_keys),
        "query_target_brand_match_count": int(len(query_target_brand_matches)),
        "query_target_brand_mention_flag": bool(query_target_brand_matches),
        "query_profile_functional_composite": weighted_jaccard(
            query_catalog_keys, profile_functional_keys, catalog_idf
        ),
        "target_profile_functional_composite": weighted_jaccard(
            target_catalog_keys, profile_functional_keys, catalog_idf
        ),
        "query_target_functional_composite": weighted_jaccard(
            query_catalog_keys, target_catalog_keys, catalog_idf
        ),
        "query_target_catalog_functional_composite": weighted_jaccard(
            query_catalog_keys, target_catalog_keys, catalog_idf
        ),
        "query_target_global_review_derived_functional_composite": weighted_jaccard(
            query_review_keys, target_review_keys, review_idf
        ),
        "target_facet_coverage_by_profile": weighted_coverage(
            target_catalog_keys, profile_functional_keys, catalog_idf
        ),
        "strict_history_temporal_ok": bool(
            history["prior_timestamp_ms"].lt(history["target_timestamp_ms"]).all()
        ) if len(history) else True,
        "strict_history_target_item_excluded_ok": bool(
            history["prior_item_id"].ne(history["target_parent_asin"]).all()
        ) if len(history) else True,
    }

    for family in FACET_FAMILIES:
        query_family_keys = keys_for_family(query_catalog_keys, family)
        profile_family_keys = keys_for_family(profile_functional_keys, family)
        target_family_keys = keys_for_family(target_catalog_keys, family)
        query_review_family_keys = keys_for_family(query_review_keys, family)
        target_review_family_keys = keys_for_family(target_review_keys, family)
        row[f"query_profile_wj__{family}"] = weighted_jaccard(
            query_family_keys, profile_family_keys, catalog_idf
        )
        row[f"target_profile_wj__{family}"] = weighted_jaccard(
            target_family_keys, profile_family_keys, catalog_idf
        )
        row[f"query_target_catalog_wj__{family}"] = weighted_jaccard(
            query_family_keys, target_family_keys, catalog_idf
        )
        row[f"query_target_global_review_derived_wj__{family}"] = weighted_jaccard(
            query_review_family_keys, target_review_family_keys, review_idf
        )
        row[f"target_profile_coverage__{family}"] = weighted_coverage(
            target_family_keys, profile_family_keys, catalog_idf
        )

    row["_profile_functional_keys_joined"] = "||".join(sorted(profile_functional_keys))
    row["_profile_brand_keys_joined"] = "||".join(sorted(profile_brand_keys))
    case_rows.append(row)

case_alignment_features = pd.DataFrame(case_rows)
require_unique_columns(case_alignment_features, "case alignment features before QCHS join")
if len(case_alignment_features) != len(queries) or case_alignment_features["case_id"].duplicated().any():
    raise RuntimeError("Case feature construction did not preserve exactly one row per case_id.")
qchs_columns = [
    "case_id",
    "prior_review_event_count",
    "prior_unique_item_count",
    "stage1_usable_prior_item_count",
    "stage1_profile_safe_prior_item_count",
    "stage1_profile_available",
    "qchs_selected_prior_item_count",
    "qchs_selected_ratio",
    "qchs_alignment_mean",
    "qchs_alignment_max",
    "qchs_attention_entropy",
    "qchs_profile_safe_facet_count",
    "qchs_brand_facet_count",
    "qchs_functional_facet_count",
    "qchs_anchor_phrase_count",
    "qchs_profile_available",
    "profile_fallback_flag",
    "profile_fallback_reason",
    "qchs_fallback_flag",
    "qchs_selected_prior_items",
    "qchs_selected_item_lineage_ok",
    "qchs_selected_count_matches",
    "qchs_retention_matches",
]
qchs_for_join = profile_diagnostics[qchs_columns].rename(columns={
    "prior_review_event_count": "qchs_input_prior_interaction_count",
    "prior_unique_item_count": "qchs_input_prior_unique_item_count",
    "stage1_usable_prior_item_count": "qchs_usable_prior_item_count",
    "stage1_profile_safe_prior_item_count": "qchs_profile_safe_prior_item_count",
    "qchs_selected_ratio": "qchs_retention_rate",
    "profile_fallback_flag": "qchs_candidate_fallback_flag",
    "profile_fallback_reason": "qchs_fallback_reason",
    "qchs_fallback_flag": "qchs_anchor_fallback_flag",
})
case_alignment_features = case_alignment_features.merge(
    qchs_for_join,
    on="case_id",
    how="left",
    validate="one_to_one",
)

if len(case_alignment_features) != len(queries):
    raise RuntimeError("QCHS join changed the case row count.")
if case_alignment_features["case_id"].duplicated().any():
    raise RuntimeError("QCHS join multiplied case rows.")
if case_alignment_features["qchs_profile_available"].isna().any():
    raise RuntimeError("QCHS join left unmatched cases.")

print("Case alignment rows:", len(case_alignment_features))
print("Case alignment columns:", len(case_alignment_features.columns))
print("Validation: one row per case_id passed")


Case alignment rows: 2288
Case alignment columns: 117
Validation: one row per case_id passed


In [10]:
# ==== Facet and Profile Coverage Summaries ====
scope_frames = [("overall", "overall", case_alignment_features)]
scope_frames.extend(
    ("regime", str(regime), group)
    for regime, group in case_alignment_features.groupby("regime", sort=False)
)

facet_summary_rows = []
for user_scope, regime, scope_df in scope_frames:
    facet_specs = [
        (
            "all_functional",
            "query_profile_functional_composite",
            "target_profile_functional_composite",
            "query_target_catalog_functional_composite",
            "query_target_global_review_derived_functional_composite",
            "target_facet_coverage_by_profile",
        )
    ]
    facet_specs.extend(
        (
            family,
            f"query_profile_wj__{family}",
            f"target_profile_wj__{family}",
            f"query_target_catalog_wj__{family}",
            f"query_target_global_review_derived_wj__{family}",
            f"target_profile_coverage__{family}",
        )
        for family in FACET_FAMILIES
    )
    for (
        facet_family,
        query_profile_column,
        target_profile_column,
        query_target_catalog_column,
        query_target_review_column,
        coverage_column,
    ) in facet_specs:
        facet_summary_rows.append({
            "category_id": CATEGORY_ID,
            "category_label": CATEGORY_LABEL,
            "user_scope": user_scope,
            "regime": regime,
            "facet_family": facet_family,
            "case_count": int(len(scope_df)),
            "query_profile_available_case_count": int(scope_df[query_profile_column].notna().sum()),
            "query_profile_mean": mean_or_nan(scope_df[query_profile_column]),
            "query_profile_median": median_or_nan(scope_df[query_profile_column]),
            "target_profile_mean": mean_or_nan(scope_df[target_profile_column]),
            "target_profile_median": median_or_nan(scope_df[target_profile_column]),
            "query_target_catalog_mean": mean_or_nan(scope_df[query_target_catalog_column]),
            "query_target_catalog_median": median_or_nan(scope_df[query_target_catalog_column]),
            "query_target_global_review_derived_mean": mean_or_nan(scope_df[query_target_review_column]),
            "query_target_global_review_derived_median": median_or_nan(scope_df[query_target_review_column]),
            "target_profile_coverage_mean": mean_or_nan(scope_df[coverage_column]),
            "target_profile_coverage_median": median_or_nan(scope_df[coverage_column]),
        })

facet_coverage_summary = pd.DataFrame(facet_summary_rows)

profile_summary_rows = []
for user_scope, regime, scope_df in scope_frames:
    active_qchs = scope_df.loc[scope_df["qchs_profile_available"]]
    profile_summary_rows.append({
        "category_id": CATEGORY_ID,
        "category_label": CATEGORY_LABEL,
        "user_scope": user_scope,
        "regime": regime,
        "case_count": int(len(scope_df)),
        "cases_with_strict_history": int(scope_df["strict_prior_interaction_count"].gt(0).sum()),
        "mean_strict_prior_interactions": mean_or_nan(scope_df["strict_prior_interaction_count"]),
        "mean_strict_prior_unique_items": mean_or_nan(scope_df["strict_prior_unique_item_count"]),
        "mean_latest_gap_days": mean_or_nan(scope_df["strict_prior_latest_gap_days"]),
        "mean_recent_180d_share": mean_or_nan(scope_df["strict_prior_recent_180d_share"]),
        "mean_profile_functional_facet_breadth": mean_or_nan(scope_df["profile_functional_facet_breadth"]),
        "mean_profile_functional_entropy_normalized": mean_or_nan(scope_df["profile_functional_entropy_normalized"]),
        "mean_profile_functional_dominant_share": mean_or_nan(scope_df["profile_functional_dominant_share"]),
        "mean_unique_prior_brand_count": mean_or_nan(scope_df["unique_prior_brand_count"]),
        "mean_dominant_brand_share": mean_or_nan(scope_df["dominant_brand_share"]),
        "mean_brand_entropy_normalized": mean_or_nan(scope_df["brand_entropy_normalized"]),
        "target_brand_seen_rate_when_available": mean_or_nan(
            scope_df.loc[scope_df["target_brand_available"], "target_brand_seen_in_prior"].astype(float)
        ),
        "qchs_profile_available_rate": mean_or_nan(scope_df["qchs_profile_available"].astype(float)),
        "qchs_candidate_fallback_rate": mean_or_nan(scope_df["qchs_candidate_fallback_flag"].astype(float)),
        "mean_qchs_selected_prior_items_active": mean_or_nan(active_qchs["qchs_selected_prior_item_count"]),
        "mean_qchs_retention_rate_active": mean_or_nan(active_qchs["qchs_retention_rate"]),
        "mean_qchs_attention_entropy_active": mean_or_nan(active_qchs["qchs_attention_entropy"]),
        "mean_qchs_alignment_active": mean_or_nan(active_qchs["qchs_alignment_mean"]),
        "strict_history_temporal_pass_rate": mean_or_nan(scope_df["strict_history_temporal_ok"].astype(float)),
        "strict_history_target_exclusion_pass_rate": mean_or_nan(
            scope_df["strict_history_target_item_excluded_ok"].astype(float)
        ),
    })

profile_and_qchs_coverage_summary = pd.DataFrame(profile_summary_rows)
require_unique_columns(facet_coverage_summary, "facet coverage summary")
require_unique_columns(profile_and_qchs_coverage_summary, "profile and QCHS coverage summary")
if facet_coverage_summary[["user_scope", "regime", "facet_family"]].duplicated().any():
    raise RuntimeError("Facet coverage summary has duplicated scope/facet rows.")
if profile_and_qchs_coverage_summary[["user_scope", "regime"]].duplicated().any():
    raise RuntimeError("Profile/QCHS coverage summary has duplicated scope rows.")
required_report_scopes = {("overall", "overall"), ("regime", "strong")}
observed_report_scopes = set(
    profile_and_qchs_coverage_summary[["user_scope", "regime"]].itertuples(
        index=False, name=None
    )
)
if not required_report_scopes.issubset(observed_report_scopes):
    raise RuntimeError("Overall and Strong-regime summaries are both required.")

print("Facet summary rows:", len(facet_coverage_summary))
print("Profile/QCHS summary rows:", len(profile_and_qchs_coverage_summary))


Facet summary rows: 40
Profile/QCHS summary rows: 5


In [11]:
# ==== Join QC, Manifest, and Exports ====
query_case_count = int(len(queries))
strict_artifact_non_cold_coverage = safe_rate(
    len(strict_artifact_case_ids & set(queries.loc[queries["regime"].ne("cold"), "case_id"])),
    int(queries["regime"].ne("cold").sum()),
)

qc_rows = [
    qc_row("query_case_id_unique", "PASS", int(queries["case_id"].nunique()), query_case_count),
    qc_row("query_user_id_unique", "PASS", int(queries["user_id"].nunique()), query_case_count),
    qc_row("qchs_case_id_unique", "PASS", int(profile_diagnostics["case_id"].nunique()), query_case_count),
    qc_row("qchs_case_universe_matches_query", "PASS", len(set(profile_diagnostics["case_id"])), query_case_count),
    qc_row("case_output_one_row_per_case_id", "PASS", int(case_alignment_features["case_id"].nunique()), query_case_count),
    qc_row("case_output_row_count", "PASS", len(case_alignment_features), query_case_count),
    qc_row("query_id_join_grain", "NOT_APPLICABLE", "not present upstream", "case_id grain", "Notebook 06 and Notebook 08 use case_id as the canonical query grain."),
    qc_row("strict_history_strictly_before_target", "PASS", int((~case_alignment_features["strict_history_temporal_ok"]).sum()), 0),
    qc_row("strict_history_target_item_excluded", "PASS", int((~case_alignment_features["strict_history_target_item_excluded_ok"]).sum()), 0),
    qc_row("training_history_multiset_subset_of_strict", "PASS", len(training_subset_violations), 0),
    qc_row("strict_artifact_reconciliation_mismatches", "PASS", len(strict_reconciliation_mismatch), 0),
    qc_row("strict_artifact_non_cold_case_coverage", "PASS", strict_artifact_non_cold_coverage, "reported", "Missing replacement cases are reconstructed from Notebook 03 raw review identifiers and timestamps."),
    qc_row("qchs_selected_items_in_training_history", "PASS", int(case_alignment_features["qchs_selected_item_lineage_ok"].sum()), query_case_count),
    qc_row("qchs_selected_count_matches_list", "PASS", int(case_alignment_features["qchs_selected_count_matches"].sum()), query_case_count),
    qc_row("qchs_retention_matches_definition", "PASS", int(case_alignment_features["qchs_retention_matches"].sum()), query_case_count),
    qc_row("query_brand_or_name_leak_flag_zero", "PASS", int(case_alignment_features["query_brand_or_name_leak_flag_upstream"].sum()), 0, "Notebook 06 outcome-independent query QC flag."),
    qc_row("query_target_brand_lexical_match_count", "REPORTED", int(case_alignment_features["query_target_brand_mention_flag"].sum()), "diagnostic only", "Literal brand-token matches are reported but not treated as leakage because common-word brand names can collide lexically."),
    qc_row("outcome_data_loaded", "PASS", len(OUTCOME_COLUMNS_LOADED), 0, "No ranks, NDCG, labels, predictions, or stage outcomes are loaded."),
    qc_row("fold_lineage", "NOT_APPLICABLE", "no model rows", "not applicable", "This measurement notebook loads no OOF prediction or model-fold artifact."),
    qc_row("candidate_source_lineage", "PASS", "Notebook 08 manifest contract", "exact query-only winner copy", "No candidate row is joined; Notebook 08 declares copy_baseline_candidates, no case-universe change, and no All Prior use for fallback."),
    qc_row("manifest_path_contracts", "PASS", 5, 5, "Notebook 03, 04, 06, and 08 output paths match configured inputs."),
]
alignment_join_qc = pd.DataFrame(qc_rows)

if case_alignment_features.columns.duplicated().any():
    raise RuntimeError("Case alignment output contains duplicate columns.")
if case_alignment_features["case_id"].duplicated().any():
    raise RuntimeError("Case alignment output contains duplicate case_id rows.")
if not case_alignment_features["qchs_selected_item_lineage_ok"].all():
    raise RuntimeError("QCHS selected-item lineage failed before export.")
if not case_alignment_features["strict_history_temporal_ok"].all():
    raise RuntimeError("Strict-history temporal validation failed before export.")
if not case_alignment_features["strict_history_target_item_excluded_ok"].all():
    raise RuntimeError("Strict-history target-item exclusion failed before export.")

manifest = {
    "notebook_name": NOTEBOOK_NAME,
    "latest_revision": f"case-level query-history coherence measurement layer ({REVISION_DATE})",
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "analysis_grain": "exactly_one_row_per_case_id",
    "primary_measure": PRIMARY_FUNCTIONAL_COMPOSITE,
    "primary_measure_scope": "strict_pre_target_all_prior_catalog_functional_profile",
    "outcome_data_used": False,
    "outcome_columns_loaded": OUTCOME_COLUMNS_LOADED,
    "weights_or_thresholds_selected_from_outcomes": False,
    "alignment_definitions": {
        "idf": "log((1 + active_catalog_item_count) / (1 + document_frequency)) + 1",
        "weighted_jaccard": "sum_idf(intersection) / sum_idf(union); NaN only when the union is empty",
        "target_facet_coverage": "sum_idf(target_intersection_profile) / sum_idf(target); NaN only when target has no eligible catalog-functional facet",
        "query_facet_match": "contiguous normalized facet-token sequence in the active synthetic query; Notebook 08 category-specific token equivalence is retained",
        "primary_functional_composite": "IDF-weighted Jaccard across all seven functional families; not an outcome-tuned arithmetic average",
        "facet_family_values": [
            "query_profile_wj",
            "target_profile_wj",
            "query_target_catalog_wj",
            "query_target_global_review_derived_wj",
            "target_profile_coverage",
        ],
    },
    "functional_facet_families": FACET_FAMILIES,
    "review_derived_semantic_mapping": REVIEW_SUFFIX_TO_FAMILY,
    "query_target_evidence_separation": {
        "catalog_derived": "Notebook 04 metadata product-functional rows only",
        "global_review_derived": "Notebook 04 historical review-derived rows mapped to a functional family",
        "unmapped_review_derived_rows": unmapped_review_derived_rows,
    },
    "history_contract": {
        "strict_profile_scope": "same-user interactions with prior_timestamp_ms < target_timestamp_ms and prior_item_id != target_parent_asin",
        "strict_profile_source": "reconstructed for the final Notebook 06 case universe from Notebook 03 raw review identifiers and timestamps",
        "strict_artifact_reconciliation": "Notebook 03 strict-history artifact is matched as a multiset wherever it covers final query cases",
        "training_safe_scope": "Notebook 03 training-safe prior artifact",
        "recent_window_days": RECENT_WINDOW_DAYS,
        "target_or_future_interaction_used": False,
    },
    "profile_distribution": {
        "event_mass": "each prior interaction contributes total mass 1 across its eligible functional facets and separately across its brand facets",
        "brand_kept_separate": True,
        "historical_review_rows_used_in_user_profile": False,
    },
    "qchs_contract": {
        "source": "Notebook 08 case-level profile diagnostics",
        "selection_recomputed_here": False,
        "selected_item_lineage": "every selected item must belong to the case training-safe prior history",
        "retention": "qchs_selected_prior_item_count / Notebook 08 input prior_unique_item_count; zero when the denominator is zero",
        "candidate_fallback": "Notebook 08 profile_fallback_flag",
        "candidate_fallback_manifest_contract": candidate_fallback_contract,
    },
    "input_paths": {
        "query_cache": str(QUERY_CACHE_PATH),
        "query_contract": str(QUERY_CONTRACT_PATH),
        "raw_review_identifiers_and_timestamps": str(RAW_REVIEWS_PATH),
        "strict_history_artifact": str(STRICT_HISTORY_PATH),
        "training_safe_history": str(TRAINING_HISTORY_PATH),
        "sampling_manifest": str(SAMPLING_MANIFEST_PATH),
        "item_facets": str(ITEM_FACETS_PATH),
        "item_docs": str(ITEM_DOCS_PATH),
        "retrieval_manifest": str(RETRIEVAL_MANIFEST_PATH),
        "profile_diagnostics": str(PROFILE_DIAGNOSTICS_PATH),
        "personalized_manifest": str(PERSONALIZED_MANIFEST_PATH),
    },
    "output_paths": {name: str(path) for name, path in OUTPUT_FILES.items()},
    "case_count": query_case_count,
    "case_feature_columns": list(case_alignment_features.columns),
    "strict_history_artifact_extra_case_count": strict_history_extra_case_count,
    "training_history_extra_case_count": training_history_extra_case_count,
    "strict_history_artifact_non_cold_case_coverage_rate": strict_artifact_non_cold_coverage,
    "validation_results": {
        "query_case_id_unique": True,
        "query_user_id_unique": True,
        "profile_diagnostics_case_id_unique": True,
        "case_universe_equal": True,
        "case_output_one_to_one": True,
        "strict_history_strictly_pre_target": True,
        "strict_history_target_excluded": True,
        "training_history_subset_of_strict": True,
        "strict_artifact_reconciled_where_available": True,
        "qchs_selected_items_training_safe": True,
        "qchs_retention_recomputed": True,
        "upstream_query_brand_or_name_leak_flag_zero": True,
        "no_outcome_data_loaded": True,
        "output_columns_unique": True,
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

case_alignment_features.to_parquet(OUTPUT_FILES["case_alignment_features"], index=False)
alignment_join_qc.to_csv(
    OUTPUT_FILES["alignment_join_qc"], index=False, encoding="utf-8-sig"
)
facet_coverage_summary.to_csv(
    OUTPUT_FILES["facet_coverage_summary"], index=False, encoding="utf-8-sig"
)
profile_and_qchs_coverage_summary.to_csv(
    OUTPUT_FILES["profile_and_qchs_coverage_summary"], index=False, encoding="utf-8-sig"
)
OUTPUT_FILES["alignment_definition_manifest"].write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

missing_outputs = [str(path) for path in OUTPUT_FILES.values() if not path.exists()]
if missing_outputs:
    raise RuntimeError(f"Required alignment outputs were not written: {missing_outputs}")

print("Preference alignment outputs written to:", OUT_DIR)
print("Primary functional composite:", PRIMARY_FUNCTIONAL_COMPOSITE)
print("Validation: PASS")


Preference alignment outputs written to: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/preference_alignment
Primary functional composite: query_profile_functional_composite
Validation: PASS


In [12]:
# ==== W2-1 FIREWALLED OUTCOME-JOIN — Step A: top-5 Promoted Alignment Lift The leak-free Alignment Body Above Keeps OUTCOME_COLUMNS_LOADED == []. Here We Load ONL the Reranked top-5 MEMBERSHIP (a Model output) and Measure profile(pre-target) <-> promoted- Item CATALOG Facet alignment. Held-out TARGET Facets Are NEVER Used (leakage guard). LightGBM = primary/headline; Transformer = Convergent check. Uplift Is Sparse (~90-95% of Cases Keep top-5 unchanged), So We Report BOTH Overall Lift AND Lift Conditional on top-5 change. ====
if OUTCOME_COLUMNS_LOADED:
    raise RuntimeError("Alignment feature body must stay outcome-free before the join step.")

_ALIGN = case_alignment_features.set_index("case_id")
_profile_func = {
    str(cid): (set(str(s).split("||")) - {""})
    for cid, s in _ALIGN["_profile_functional_keys_joined"].items()
}

RERANK_SOURCES = {
    ("lightgbm", "S2-Q"): PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/lightgbm_no_prior/reranked_candidates.parquet",
    ("lightgbm", "S2-P"): PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/lightgbm/reranked_candidates.parquet",
    ("transformer", "S2-Q"): PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/transformer_no_prior/reranked_candidates.parquet",
    ("transformer", "S2-P"): PROJECT_ROOT / "outputs/stage2_nonpersonalized_rerank/transformer/reranked_candidates.parquet",
}
RERANK_DEPTH = 1000  # report depth 700->1000 (2026-07-23); reranking always top-1000
TOPK = 5

def _resolve_col(cols, aliases):
    lut = {str(c).strip().lower(): c for c in cols}
    for a in aliases:
        if a.lower() in lut:
            return lut[a.lower()]
    return None

def _top5_by_case(path):
    df = pd.read_parquet(path)
    id_col = _resolve_col(df.columns, ["candidate_parent_asin", "candidate_item_id"])
    rank_col = _resolve_col(df.columns, ["rerank_rank", "gt_rank", "new_rank", "candidate_rank"])
    depth_col = _resolve_col(df.columns, ["candidate_pool_depth", "pool_depth"])
    if id_col is None or rank_col is None:
        raise RuntimeError(f"reranked_candidates lacks id/rank columns: {path}")
    if depth_col is not None:
        df = df.loc[pd.to_numeric(df[depth_col], errors="raise").astype(int).eq(RERANK_DEPTH)]
    df = df.loc[pd.to_numeric(df[rank_col], errors="raise").astype(int).le(TOPK)]
    out = {}
    for cid, g in df.groupby("case_id", sort=False):
        out[str(cid)] = [str(x) for x in g.sort_values(rank_col, kind="mergesort")[id_col].tolist()]
    return out

def _promoted_keys(asins):
    keys = set()
    for a in asins:
        keys |= catalog_keys_by_item.get(a, set())
    return keys

def _boot_ci(values, seed=42, iters=10000):
    v = np.asarray([x for x in values if x is not None and np.isfinite(x)], dtype=float)
    if v.size == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(int(seed))
    means = v[rng.integers(0, v.size, size=(int(iters), v.size))].mean(axis=1)
    lo, hi = np.quantile(means, [0.025, 0.975])
    return (float(lo), float(hi))

FAMILIES_HEADLINE = [f for f in FACET_FAMILIES if f not in {"claim_constraint", "sensory"}]  # noise -> parquet only
_topk_rows = []
for model in ["lightgbm", "transformer"]:
    top_q = _top5_by_case(RERANK_SOURCES[(model, "S2-Q")])
    top_p = _top5_by_case(RERANK_SOURCES[(model, "S2-P")])
    cases = [c for c in _ALIGN.index.astype(str) if c in top_q and c in top_p]
    changed = {c: (set(top_q[c]) != set(top_p[c])) for c in cases}
    for family in ["functional_composite", *FACET_FAMILIES]:
        for subset, sel in [
            ("overall", cases),
            ("conditional_on_top5_change", [c for c in cases if changed[c]]),
        ]:
            dq, dp, dd = [], [], []
            for c in sel:
                pk = _profile_func.get(c, set())
                if family == "functional_composite":
                    lk_q, lk_p, rk = _promoted_keys(top_q[c]), _promoted_keys(top_p[c]), pk
                    wq = weighted_jaccard(lk_q, rk, catalog_idf)
                    wp = weighted_jaccard(lk_p, rk, catalog_idf)
                else:
                    wq = weighted_jaccard(keys_for_family(_promoted_keys(top_q[c]), family), keys_for_family(pk, family), catalog_idf)
                    wp = weighted_jaccard(keys_for_family(_promoted_keys(top_p[c]), family), keys_for_family(pk, family), catalog_idf)
                dq.append(wq); dp.append(wp); dd.append(wp - wq)
            lo, hi = _boot_ci(dd, seed=42)
            _topk_rows.append({
                "category_id": CATEGORY_ID,
                "model_family": model,
                "model_role": "primary_headline" if model == "lightgbm" else "convergent_check",
                "facet_family": family,
                "subset": subset,
                "in_headline_table": bool(family in (["functional_composite"] + FAMILIES_HEADLINE)),
                "n_cases": int(len(dd)),
                "mean_wj_S2Q": float(np.mean(dq)) if dq else float("nan"),
                "mean_wj_S2P": float(np.mean(dp)) if dp else float("nan"),
                "mean_delta": float(np.mean(dd)) if dd else float("nan"),
                "delta_ci_low": lo,
                "delta_ci_high": hi,
                "target_facets_used": False,
            })
alignment_topk_lift_summary = pd.DataFrame(_topk_rows)
alignment_topk_lift_summary.to_csv(OUT_DIR / "alignment_topk_lift_summary.csv", index=False, encoding="utf-8-sig")
try:
    display(alignment_topk_lift_summary)
except NameError:
    print(alignment_topk_lift_summary)


,category_id,model_family,model_role,facet_family,subset,in_headline_table,n_cases,mean_wj_S2Q,mean_wj_S2P,mean_delta,delta_ci_low,delta_ci_high,target_facets_used
0,face,lightgbm,primary_headline,functional_composite,overall,True,2288,0.106163,0.107876,0.001713,0.000403,0.003039,False
1,face,lightgbm,primary_headline,functional_composite,conditional_on_top5_change,True,1684,0.141424,0.143752,0.002327,0.000569,0.004104,False
2,face,lightgbm,primary_headline,category_or_product_type,overall,True,2288,0.206231,0.209543,0.003313,-0.000367,0.006965,False
3,face,lightgbm,primary_headline,category_or_product_type,conditional_on_top5_change,True,1684,0.274522,0.279023,0.004501,-0.000377,0.009456,False
4,face,lightgbm,primary_headline,form_texture,overall,True,2288,NaN,NaN,NaN,-0.004965,0.001336,False
5,face,lightgbm,primary_headline,form_texture,conditional_on_top5_change,True,1684,0.161677,0.159263,-0.002414,-0.006818,0.001856,False
6,face,lightgbm,primary_headline,ingredient_or_composition,overall,True,2288,NaN,NaN,NaN,-0.003415,0.001279,False
7,face,lightgbm,primary_headline,ingredient_or_composition,conditional_on_top5_change,True,1684,NaN,NaN,NaN,-0.004463,0.001615,False
8,face,lightgbm,primary_headline,need_benefit_concern,overall,True,2288,0.165835,0.169000,0.003165,0.000549,0.005893,False
9,face,lightgbm,primary_headline,need_benefit_concern,conditional_on_top5_change,True,1684,0.221197,0.225497,0.004300,0.000694,0.007991,False


In [13]:
# ==== W2-1 FIREWALLED OUTCOME-JOIN — Step B: Alignment -> Realized Gain (WITHIN-REGIME tertile) per-case Delta NDCG@5 (RankP - Base) from NB14 canonical, Joined to the FROZEN leak-free Input alignment. Tertiles Are Cut WITHIN Each non-cold Regime Because the Alignment Axis Is Confounded with Regime (target_profile_coverage Rises Monotonically cold->strong); Global Tertiles Would Let Regime Masquerade As an Alignment effect. Cold Degenerates to alignment~0 and Is Reported As a Floor (not tertiled). Functional Composite and Brand Are Each Tertiled Within regime. LightGBM Is the carrier. ====
NB14_PRIMARY_PATH = PROJECT_ROOT / "outputs/pipeline_aggregate/pipeline_canonical_primary_ndcg5_depth1000.parquet"
CARRIER_FAMILY = "lightgbm"
BRAND_ALIGNMENT_COLUMN = "target_brand_prior_interaction_share"

NB14_RAW_FALLBACK_PATH = PROJECT_ROOT / "outputs/pipeline_aggregate/pipeline_canonical_per_case_metrics.parquet"

_nb14_source_path = NB14_PRIMARY_PATH if NB14_PRIMARY_PATH.exists() else NB14_RAW_FALLBACK_PATH
if not _nb14_source_path.exists():
    raise FileNotFoundError(
        f"Missing NB14 primary and raw fallback: {NB14_PRIMARY_PATH}, {NB14_RAW_FALLBACK_PATH}"
    )

print("Using NB14 source:", _nb14_source_path)
_p = pd.read_parquet(_nb14_source_path)

if "analysis_method_family" not in _p.columns:
    if "method_family" not in _p.columns:
        raise RuntimeError("NB14 source has neither analysis_method_family nor method_family.")
    _p["analysis_method_family"] = _p["method_family"].astype(str)

_p = _p.loc[
    _p["analysis_method_family"].astype(str).eq(CARRIER_FAMILY)
    & _p["metric_name"].astype(str).eq("NDCG")
    & pd.to_numeric(_p["metric_cutoff"], errors="raise").astype(int).eq(5)
    & pd.to_numeric(_p["candidate_pool_depth"], errors="raise").astype(int).eq(1000)
    & _p["stage_condition"].astype(str).isin(["P2-Q", "P2-P"])
].copy()
_p["case_id"] = _p["case_id"].astype(str)
_wide = _p.pivot_table(index="case_id", columns="stage_condition", values="metric_value", aggfunc="first")
_col_q = "P2-Q" if "P2-Q" in _wide.columns else None
_col_p = "P2-P" if "P2-P" in _wide.columns else None
if _col_q is None or _col_p is None:
    raise RuntimeError(f"NB14 primary lacks Base/RankP columns: {list(_wide.columns)}")
_wide = _wide.assign(delta_ndcg5=_wide[_col_p] - _wide[_col_q]).reset_index()

_align_cols = ["case_id", "regime", "query_profile_functional_composite", BRAND_ALIGNMENT_COLUMN]
_align = case_alignment_features[_align_cols].copy()
_align["case_id"] = _align["case_id"].astype(str)
_join = _align.merge(_wide[["case_id", "delta_ndcg5"]], on="case_id", how="inner", validate="one_to_one")

def _within_regime_tertiles(frame, value_col, seed=42):
    rows = []
    cold = frame.loc[frame["regime"].astype(str).eq("cold")]
    if len(cold):
        lo, hi = _boot_ci(cold["delta_ndcg5"].tolist(), seed=seed)
        rows.append({"regime": "cold", "tertile": "floor_all_cold", "n": int(len(cold)),
                     "mean_delta_ndcg5": float(cold["delta_ndcg5"].mean()), "ci_low": lo, "ci_high": hi})
    for regime in [r for r in REGIME_ORDER if r != "cold"]:
        g = frame.loc[frame["regime"].astype(str).eq(regime)].copy()
        if len(g) < 3:
            continue
        try:
            g["_tertile"] = pd.qcut(g[value_col].rank(method="first"), 3, labels=["T1_low", "T2_mid", "T3_high"])
        except ValueError:
            continue
        for t, sub in g.groupby("_tertile", observed=True):
            lo, hi = _boot_ci(sub["delta_ndcg5"].tolist(), seed=seed)
            rows.append({"regime": regime, "tertile": str(t), "n": int(len(sub)),
                         "mean_delta_ndcg5": float(sub["delta_ndcg5"].mean()), "ci_low": lo, "ci_high": hi})
    return rows

_stratum_rows = []
for axis, col in [("functional_composite", "query_profile_functional_composite"), ("brand", BRAND_ALIGNMENT_COLUMN)]:
    sub = _join.dropna(subset=[col, "delta_ndcg5"]).copy()
    for r in _within_regime_tertiles(sub, col, seed=42):
        r.update({"category_id": CATEGORY_ID, "carrier_family": CARRIER_FAMILY, "alignment_axis": axis})
        _stratum_rows.append(r)
alignment_gain_by_stratum = pd.DataFrame(_stratum_rows)
alignment_gain_by_stratum.to_csv(OUT_DIR / "alignment_gain_by_stratum.csv", index=False, encoding="utf-8-sig")
try:
    display(alignment_gain_by_stratum)
except NameError:
    print(alignment_gain_by_stratum)


Using NB14 source: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/pipeline_aggregate/pipeline_canonical_per_case_metrics.parquet


,regime,tertile,n,mean_delta_ndcg5,ci_low,ci_high,category_id,carrier_family,alignment_axis
0,cold,floor_all_cold,572,0.000000,0.000000,0.000000,face,lightgbm,functional_composite
1,weak,T1_low,191,0.038020,0.014097,0.065445,face,lightgbm,functional_composite
2,weak,T2_mid,190,0.017857,-0.009376,0.045686,face,lightgbm,functional_composite
3,weak,T3_high,191,0.015415,0.001092,0.031030,face,lightgbm,functional_composite
4,moderate,T1_low,191,-0.002255,-0.018127,0.011926,face,lightgbm,functional_composite
5,moderate,T2_mid,190,0.014474,0.000167,0.033229,face,lightgbm,functional_composite
6,moderate,T3_high,191,0.024475,0.006575,0.045353,face,lightgbm,functional_composite
7,strong,T1_low,191,0.019466,0.002553,0.039128,face,lightgbm,functional_composite
8,strong,T2_mid,190,0.003227,-0.010954,0.018056,face,lightgbm,functional_composite
9,strong,T3_high,191,0.007168,-0.008530,0.023719,face,lightgbm,functional_composite


In [14]:
# ==== W2-1 Firewall Re-declaration and Delete/Reduce Documentation ====
# W2-1 firewall re-declaration + delete/reduce documentation
_w2_manifest = {
    "notebook": NOTEBOOK_NAME,
    "alignment_features_outcome_free": (len(OUTCOME_COLUMNS_LOADED) == 0),
    "outcome_loaded_for_association_only_in_downstream_cells": True,
    "stepA_topk_lift_uses_target_facets": False,
    "stepA_source": "profile_pretarget_keys + promoted_item_catalog_facets",
    "stepA_reports": ["overall", "conditional_on_top5_change"],
    "stepB_tertile_policy": "within_regime_noncold; cold reported as floor; functional & brand axes",
    "carrier_primary": "lightgbm",
    "carrier_convergent": "transformer",
    "headline_excluded_noise_families": ["claim_constraint", "sensory"],
    "headline_excluded_qchs_internal": ["qchs_attention_entropy", "qchs_anchor_phrase_count", "qchs_alignment_max"],
    "note_transformer_skincare_may_be_null": "Transformer prior effect is category-split (Supplements sig / Skincare non-sig); a null Skincare lift is convergent with the known effect, not a defect.",
    "outputs": {
        "alignment_topk_lift_summary": str(OUT_DIR / "alignment_topk_lift_summary.csv"),
        "alignment_gain_by_stratum": str(OUT_DIR / "alignment_gain_by_stratum.csv"),
    },
}
(OUT_DIR / "w2_preference_alignment_outcome_join_manifest.json").write_text(
    json.dumps(_w2_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("W2-1 outcome-join complete:", _w2_manifest["outputs"])


W2-1 outcome-join complete: {'alignment_topk_lift_summary': '/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/preference_alignment/alignment_topk_lift_summary.csv', 'alignment_gain_by_stratum': '/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/preference_alignment/alignment_gain_by_stratum.csv'}
